# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library, referencing all data entities strictly by their `@id`.

### Dataset Source
The dataset is defined by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will explore the metadata to enumerate all available record sets and their fields. All references will be made using the `@id` property, following Croissant conventions.

In [ ]:
# List all record sets and their fields using the Croissant schema structure.
from pprint import pprint
# Helper function to represent entities by their @id.
def safe_get(obj, key):
    return getattr(obj, key, None) if hasattr(obj, key) else obj.get(key, None)

print('Record sets in the dataset:')
record_set_ids = []
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets  # Newest mlcroissant versions use 'record_sets'
elif hasattr(metadata, 'record_set'):
    record_sets = metadata.record_set   # Croissant 1.0 often uses singular 'record_set'
else:
    record_sets = []

for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id if hasattr(rs, 'id') else rs['@id']}")
    record_set_ids.append(rs.id if hasattr(rs, 'id') else rs['@id'])
    # List fields in each record set
    if hasattr(rs, 'field') and rs.field:
        print("  Fields:")
        for f in rs.field:
            # Field object or reference
            if hasattr(f, 'id'):
                print(f"    - {f.name} (@id: {f.id}) - type: {getattr(f, 'data_type', 'N/A')}")
            elif isinstance(f, dict):
                print(f"    - {f.get('name', f.get('@id', ''))} (@id: {f.get('@id','')}) - type: {f.get('dataType','N/A')}")
        print()
print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Below, we select a record set by its `@id`, as discovered above. You may adjust `selected_record_set_id` to any available record set.

In [ ]:
# Use the first record set for demonstration; replace with another @id as needed
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
else:
    raise Exception("No record set found in this dataset.")

print(f"Loading records from RecordSet with @id: {selected_record_set_id}")
records = list(dataset.records(record_set=selected_record_set_id))

df = pd.DataFrame(records)
print('Fields (@id):', df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We continue to reference fields by their Croissant `@id`.

Choose a numeric field for demonstration (edit as needed using the column list above):

In [ ]:
# Try to select a relevant numeric field based on the available fields
numeric_field_id = None
numeric_types = ['Number', 'Float', 'Integer']
field_map = {}
chosen_group_field_id = None

# Find the record set object
selected_record_set = None
for rs in (getattr(metadata, 'record_sets', []) or getattr(metadata, 'record_set', [])):
    if (hasattr(rs, 'id') and rs.id == selected_record_set_id) or (isinstance(rs, dict) and rs.get('@id') == selected_record_set_id):
        selected_record_set = rs
        break

if selected_record_set is not None and hasattr(selected_record_set, 'field'):
    for f in selected_record_set.field:
        fid = f.id if hasattr(f,'id') else f.get('@id','')
        dtype = getattr(f, 'data_type', f.get('dataType', ''))
        field_map[fid] = f
        if numeric_field_id is None and any(tp in dtype for tp in numeric_types):
            # Choose the first numeric type found
            numeric_field_id = fid
        if chosen_group_field_id is None and 'group' in (getattr(f, 'name', '') or f.get('name', '')).lower():
            chosen_group_field_id = fid

if not numeric_field_id:
    # If not found, just take the first field that looks numeric based on values
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

if not numeric_field_id:
    raise Exception("No numeric field found in this RecordSet for demonstration.")

print(f"Using numeric field: {numeric_field_id}")

# Filtering
threshold = df[numeric_field_id].quantile(0.75) if not pd.isnull(df[numeric_field_id]).all() else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (upper quartile):")
print(filtered_df.head())

# Normalization (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a group field for grouping analysis (if available and categorical)
if not chosen_group_field_id:
    for col in df.columns:
        if df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df[col]):
            chosen_group_field_id = col
            break

if chosen_group_field_id:
    grouped_df = filtered_df.groupby(chosen_group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {chosen_group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We visualize the distribution of the selected numeric field and, if available, its distribution by a categorical group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id} (by @id)")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group field available, boxplot
if chosen_group_field_id:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=chosen_group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {chosen_group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to reference, load, and process a FAIR^2 dataset using the Croissant `@id` structure with `mlcroissant`. We performed exploratory data analysis and basic visualizations. For more advanced analyses or to work with additional record sets, adjust the referenced `@id`s and fields as needed. All actions here are directly traceable to the dataset schema and are fully FAIR-compliant.